# 상추의 생육 환경 생성 AI 경진대회

##[배경 및 주제]
https://dacon.io/competitions/official/236033/overview/description


### 배경
4차 산업혁명 시대를 맞아 농업 분야에서도 인공지능(AI) 기술이 널리 사용되어 IT 기술을 동원한 스마트팜 등 보다 효율적인 작물 재배가 가능해지고 있습니다.  

### 주제
생육 환경 생성 AI 모델 결과를 바탕으로 상추의 일별 최대 잎 중량을 도출할 수 있는 최적의 생육 환경 조성

### 데이터
**train_input [폴더] - 총 28개 상추 케이스**

DAT : 생육 일 (0~27일차)

obs_time : 측정 시간

상추 케이스 별 환경 데이터 (1시간 간격)


**train_target [폴더] - 총 28개 상추 케이스**

DAT : 생육 일 (1~28일차)

predicted_weight_g : 일별 잎 중량


**test_input [폴더] - 총 5개 상추 케이스**

DAT : 생육 일 (0~27일차)

obs_time : 측정 시간

상추 케이스 별 환경 데이터 (1시간 간격)


**test_target [폴더] - 총 5개 상추 케이스**

DAT : 생육 일 (1~28일차)

predicted_weight_g : 일별 예측한 잎 중량

제출을 위한 양식으로 target에 해당되는 predicted_weight_g의 값은 모두 0으로 가려져있습니다.


## 코드 흐름

###(1) EDA
-  각 feature 들이 밤낮에 따라 변화o

- 피어슨 상관 계수를 통해 feature들을 일별 평균하여 상관관계 출력

###(2) 전처리
- 온도, 습도, 이산화탄소를 각각 곱한 파생변수 생성
- 특정 시간 광량의 존재여부에 따른 '광량여부'칼럼 생성
- 모든 case를 flatten 하여 일별 데이터 생성
- target 변수: 산추의 생장치로 변경
- 이상치 처리
  * 내부온도,내부 습도 0 -> 평균으로 대치
  * 시간당 분무량이 음수인 케이스 -> 0으로 대체
  * 시간당 백색,적색, 청색광량이 음수인 케이스-> 최빈값으로 대체

### (3) 예측 모델 실험
- Standard Scaler 이용해서 표준화 진행
- XGBoost, LGBM 사용
- Extra tree <- 랜덤포레스트보다 더 많은 무작위성을 주입

### (4) 생성 모델
- 모델 선정 : conditional VAE 모델 선정

- 학습 데이터 -> x, condition나눔

- 모델 구조 -> convolution layer로 특징 추출, fully
connected layer사용

- loss function : reconstruction loss+value_loo 사용

###(5) 결론
-내부 온도와 이산화탄소를 균일하게 24시간 하는 것보다 특정 시간대에 높이는 것이 성장에 유리함
-모델에 대한 각 피처의 영향량을 잘 반영하면 생성환경을 다양하게 생성할 수 있음



##[차별점 및 배운점]
### 차별점
- 예측 모델을 실험을 할때 보통 xgboost, lightgbm이 성능이 높다는 것을 알고 있는데 extra tree를 사용하면 랜덤포레스터보다 더 무작위성을 주입하여 사용이 가능하다는 사실을 알게 되었고, 추후에 활용해 보아야겠다고 생각했다.

### 배울점
- conditional vae 사용 방법 :주로 입력 데이터와 함께 적용되어, 생성된 출력이 특정 조건을 만족하도록 하며,  훈련 데이터에서 원하는 특성을 재현하거나 특정한 출력을 생성하는 데 사용된다고 한다.
- condition을 부여한다는 것이 label의 정보를 알고 있으면 encoder와 decoder을 사용하며 구현시 albel의 정보를 추가해주면 된다는 것을 알게 되었다.